# Three APTOS training runs using the paper configuration

This notebook queues **three independent APTOS-only fine-tuning runs** on the recovered original split:
2,929 training images, 366 validation images and 367 test images. Every run starts from the same
cached **base model**, initializes a new LoRA adapter, and uses the same seeds and settings.
It never continues from the project's presented adapter or another run's adapter.

Each run selects its best checkpoint using **APTOS validation loss**, saves its own `best_lora/`,
then evaluates the 367 APTOS test images using the original full-answer likelihood scorer.
External testing on all 1,057 Messidor images is also enabled; Messidor never enters training or validation.
Test results do not influence checkpoint selection. No mixed-dataset or offline-augmentation experiment is used.

This is a fixed-seed repeatability experiment, not a three-seed performance estimate.
The existing `train_aptos&messidor.ipynb` and the three existing training/evaluation scripts are left untouched.

## Environment and how to run

Use **Python (afterimage_env)** in VS Code/Jupyter. From Anaconda Prompt at the repository root:

```bat
conda activate afterimage_env
python -m pip install notebook ipykernel
python -m ipykernel install --user --name afterimage_env --display-name "Python (afterimage_env)"
```

The project model dependencies must already be installed. Optional plots use `matplotlib`
(`python -m pip install matplotlib` if needed); all predictions and numeric reports work without it.
This notebook never installs or upgrades model packages. Stop/unload other GPU model servers before running.

Run the cells from top to bottom. The kernel only prepares data and launches subprocesses; each GPU process
exits when finished. Do not run both notebooks simultaneously on the same GPU. For unattended runs, keep
the computer powered, disable sleep for that session, and keep Jupyter/the notebook kernel running.

## Paper configuration and seed details

| Setting | Value |
| --- | --- |
| Base | Qwen3-VL-8B Instruct, 4-bit |
| Image dimensions | 512 × 512, already CLAHE processed |
| Per-device training batch / accumulation | **4 / 2** (effective batch 8) |
| Learning rate / scheduler / warmup | **5e-4 / linear / 20 steps** |
| Maximum epochs / sequence length | **50 / 2048 tokens** |
| Optimizer / weight decay | **AdamW-8bit / 0.001** |
| LoRA rank / alpha / dropout / bias | **16 / 16 / 0.05 / none** |
| Gradient checkpointing | Enabled |
| Class imbalance | Inverse-frequency weighted sampling, with replacement |
| Evaluation / saving | Every epoch; retain two trainer checkpoints |
| Early stopping | Minimum validation loss; patience **2**, threshold **0.0** |

The original script defaults its **Trainer seed to 42** and omits the Unsloth LoRA initialization
seed argument. This workflow preserves that call behavior. The installed Unsloth default is **3407**;
it is checked and recorded for every run. The paper does not explicitly record that historical LoRA seed,
so 3407 is identified as the current verified library default, not an independently verified paper value.
The split is read from the recovered CSV rather than rebuilt using a seed.

The original script also leaves validation batch size at the Transformers default **8**, which is made
explicit here. Training uses BF16 mixed precision on the verified RTX 5070 Ti; evaluation follows the
reference's forced FP16 setup. These defaults are recorded separately from the explicitly reported paper values.
GPU deterministic algorithms are not forced because the original script did not force them.

The original software stack was Unsloth 2026.2.1 / Torch 2.10.0 / Transformers 5.3.0.dev0.
This notebook uses your existing `afterimage_env`, records its versions, and does not downgrade packages.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid
import pandas as pd
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "finetuning/Qwen3/paper_reproduction.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter inside the Afterimage checkout.")
QWEN_DIR = REPO_ROOT / "finetuning/Qwen3"
if str(QWEN_DIR) not in sys.path:
    sys.path.insert(0, str(QWEN_DIR))
import paper_reproduction as reproduction

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current Python: {sys.executable}")

ORIGINAL_APTOS_SPLIT = Path(r"C:\Users\moht2\Downloads\aptos_dataset_80_10_10.csv")
DATA_ROOT = Path(r"C:\Users\moht2\Downloads\DR_Aptos_Messidor_Only_Full_Dataset\Res_512")
IMAGE_MAPPING_CSV = DATA_ROOT / "Split_80_10_10/dataset_80_10_10.csv"
IMAGE_DIR = DATA_ROOT / "Aptos_messidor_external_CLAHE"
WORK_ROOT = QWEN_DIR / "artifacts/paper_reproduction"
VERIFY_IMAGES = True
print("Python:", sys.executable)
print("Original split:", ORIGINAL_APTOS_SPLIT)
print("Generated outputs:", WORK_ROOT)


# Keep this name when restarting an interrupted batch. Choose a NEW name for a new set of three runs.
BATCH_ID = "aptos_paper_same_seed_three_runs_001"
BATCH_DIR = WORK_ROOT / "training_batches" / BATCH_ID
INCLUDE_EXTERNAL_MESSIDOR = True
print("Three-run batch:", BATCH_DIR)

Python: c:\Users\moht2\.conda\envs\afterimage_env\python.exe
Original split: C:\Users\moht2\Downloads\aptos_dataset_80_10_10.csv
Generated outputs: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction
Three-run batch: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\training_batches\aptos_paper_same_seed_three_runs_001


## Verify the original split and available model snapshot

In [2]:
DATASET_MANIFEST = reproduction.prepare_data(
    ORIGINAL_APTOS_SPLIT, IMAGE_MAPPING_CSV, IMAGE_DIR, WORK_ROOT, verify_images=VERIFY_IMAGES,
)
manifest = json.loads(DATASET_MANIFEST.read_text(encoding="utf-8"))
counts = pd.DataFrame([
    {"partition": split, "source": "APTOS", "images": manifest["counts"][split]}
    for split in ("train", "val", "test")
] + [{"partition": "external test", "source": "Messidor", "images": manifest["counts"]["messidor_external"]}])
display(counts)
display(pd.DataFrame(reproduction.APTOS_COUNTS, index=pd.Index(range(5), name="grade")).T)
print("Original APTOS test: 367 images. Messidor is a separate external cohort of 1,057 images.")
print("Original CSVs and image files have not been modified.")

Verified 1000/4719 image files
Verified 2000/4719 image files
Verified 3000/4719 image files
Verified 4000/4719 image files
Verified 4719/4719 image files


,partition,source,images
0,train,APTOS,2929
1,val,APTOS,366
2,test,APTOS,367
3,external test,Messidor,1057


grade,0,1,2,3,4
train,1444,296,799,154,236
val,180,37,100,20,29
test,181,37,100,19,30


Original APTOS test: 367 images. Messidor is a separate external cohort of 1,057 images.
Original CSVs and image files have not been modified.


In [3]:
# Pin the exact local base snapshot used in the successful saved-adapter check.
# local_files_only=True prevents an unattended download or silent model revision change.
BASE_SNAPSHOT = reproduction.resolve_base_snapshot()
print("Cached base snapshot:", BASE_SNAPSHOT)
display(pd.Series(reproduction.package_versions(), name="installed version"))

c:\Users\moht2\.conda\envs\afterimage_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cached base snapshot: C:\Users\moht2\.cache\huggingface\hub\models--unsloth--Qwen3-VL-8B-Instruct-unsloth-bnb-4bit\snapshots\b5b904c3fcdc7541adf2a2bb219b0ed95288c794


torch           2.14.1+cu130
torchvision     0.29.1+cu130
unsloth           2026.10.10
unsloth_zoo       2026.10.10
transformers          5.17.0
trl                   1.15.0
peft                  0.21.2
datasets               4.8.5
numpy                  2.4.6
pandas                 3.0.6
bitsandbytes          0.50.2
Name: installed version, dtype: str

## Freeze the configuration before starting

The same configuration file feeds all three processes. The source/data/package fingerprints prevent a
restart from silently mixing settings. Batch size is **4**; this workflow will not automatically reduce it
or alter any other paper setting after a failure. Review the table below before leaving the run unattended.

In [4]:
configuration = reproduction.make_training_config(DATASET_MANIFEST, BASE_SNAPSHOT)
BATCH_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = BATCH_DIR / "paper_training_configuration.json"
if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text(encoding="utf-8")) != configuration:
    raise ValueError("This batch name already has different settings. Choose a new BATCH_ID.")
reproduction.write_json(CONFIG_PATH, configuration)
display(pd.Series(configuration["training"], name="same value in all three runs"))
display(pd.DataFrame([
    {"run": i, "trainer_seed": 42, "Unsloth_default_seed": 3407,
     "adapter": str(BATCH_DIR / f"run_{i:02d}" / "best_lora")}
    for i in range(1, 4)
]))
print("Configuration:", CONFIG_PATH)

epochs                                              50
batch_size                                           4
grad_accum                                           2
lr                                              0.0005
warmup_steps                                        20
max_length                                        2048
weight_decay                                     0.001
optim                                       adamw_8bit
scheduler                                       linear
lora_r                                              16
lora_alpha                                          16
lora_dropout                                      0.05
bias                                              none
gradient_checkpointing                            True
img_size                                           512
load_in_4bit                                      True
eval_strategy                                    epoch
save_strategy                                    epoch
save_total

,run,trainer_seed,Unsloth_default_seed,adapter
0,1,42,3407,c:\Users\moht2\Documents\Afterimage\finetuning...
1,2,42,3407,c:\Users\moht2\Documents\Afterimage\finetuning...
2,3,42,3407,c:\Users\moht2\Documents\Afterimage\finetuning...


Configuration: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\training_batches\aptos_paper_same_seed_three_runs_001\paper_training_configuration.json


## Launch all three runs sequentially

Execute this cell once, then leave the notebook and computer running. Each training subprocess exits
before evaluation starts, and each evaluation process exits before the next training run.

Completed adapters are checked and preserved when you restart this same batch. An interrupted run can
resume from its most recent complete trainer checkpoint, including optimizer, scheduler and RNG state.
Early-stopping callback state is restored. If only incomplete checkpoints exist, that run reports a failure
instead of silently pretending to resume. All run failures are recorded; the queue continues with the next
independent run. A user interrupt stops the queue. Every successfully trained adapter is saved **before** testing.

In [5]:
batch_results = reproduction.run_three(
    CONFIG_PATH, BATCH_DIR, include_external=INCLUDE_EXTERNAL_MESSIDOR,
)
display(pd.DataFrame(batch_results))
failed = [row for row in batch_results if row["status"] != "complete"]
if failed:
    print(f"{len(failed)} run(s) need attention. Inspect their failure.json and logs; completed adapters are preserved.")
else:
    print("All three independent adapters and their evaluation reports are saved.")


===== RUN 1/3: same seeds, fresh base, original APTOS split =====
c:\Users\moht2\.conda\envs\afterimage_env\python.exe -u c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\paper_reproduction.py train --config C:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\training_batches\aptos_paper_same_seed_three_runs_001\paper_training_configuration.json --run-dir C:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\training_batches\aptos_paper_same_seed_three_runs_001\run_01
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1105 08:34:58.551000 2080 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1105 08:34:58.599000 2080 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is

,run,run_dir,adapter_dir,status,best_epoch,best_eval_loss,adapter_sha256,epochs_completed,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy,messidor_accuracy,messidor_qwk,messidor_macro_ovr_auc
0,1,C:\Users\moht2\Documents\Afterimage\finetuning...,C:\Users\moht2\Documents\Afterimage\finetuning...,complete,3.0,0.008101,caa3918d9a0dee9e76fe75bb0250833d9cb188e44d0959...,5.0,0.773842,0.866584,0.942846,0.613749,0.643718,0.446547,0.561342,0.805607
1,2,C:\Users\moht2\Documents\Afterimage\finetuning...,C:\Users\moht2\Documents\Afterimage\finetuning...,complete,2.0,0.007336,5ef24c4e43d6c50546fafb73b1d4034d42ccff0ed06fdf...,4.0,0.806540,0.891526,0.936183,0.623312,0.602476,0.505203,0.552775,0.773197
2,3,C:\Users\moht2\Documents\Afterimage\finetuning...,C:\Users\moht2\Documents\Afterimage\finetuning...,complete,3.0,0.008312,66c7d56a4f9cb9f5e2375c961ea2e23bd6b3a6d0d1792a...,5.0,0.771117,0.872970,0.931582,0.631052,0.661158,0.489120,0.584465,0.796058


All three independent adapters and their evaluation reports are saved.


## Compare the three runs

In [6]:
batch_results = json.loads((BATCH_DIR / "batch_results.json").read_text(encoding="utf-8"))
results = pd.DataFrame(batch_results)
display(results)
successful = results.loc[results["status"].eq("complete")].copy()
metric_columns = ["accuracy", "qwk", "macro_ovr_auc", "macro_f1", "balanced_accuracy"]
if not successful.empty:
    comparison = successful[["run", "best_epoch", "epochs_completed", "best_eval_loss", *metric_columns]].copy()
    display(comparison)
    comparison.to_csv(BATCH_DIR / "aptos_run_comparison.csv", index=False)
    # Descriptive variation under the SAME seeds/settings; no best-by-test selection.
    variation = successful[metric_columns].agg(["count", "mean", "std", "min", "max"])
    display(variation)
    variation.to_csv(BATCH_DIR / "aptos_repeatability_summary.csv")
    display(successful[["run", "adapter_dir", "adapter_sha256"]])
    print("Paper reference: accuracy 0.8174; QWK 0.9292; macro ROC-AUC 0.9488.")
    print("Compare all completed runs; test metrics were not used to select checkpoints.")
if len(successful) != 3:
    print(f"Only {len(successful)}/3 runs completed; this is not yet a complete three-run comparison.")

,run,run_dir,adapter_dir,status,best_epoch,best_eval_loss,adapter_sha256,epochs_completed,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy,messidor_accuracy,messidor_qwk,messidor_macro_ovr_auc
0,1,C:\Users\moht2\Documents\Afterimage\finetuning...,C:\Users\moht2\Documents\Afterimage\finetuning...,complete,3.0,0.008101,caa3918d9a0dee9e76fe75bb0250833d9cb188e44d0959...,5.0,0.773842,0.866584,0.942846,0.613749,0.643718,0.446547,0.561342,0.805607
1,2,C:\Users\moht2\Documents\Afterimage\finetuning...,C:\Users\moht2\Documents\Afterimage\finetuning...,complete,2.0,0.007336,5ef24c4e43d6c50546fafb73b1d4034d42ccff0ed06fdf...,4.0,0.806540,0.891526,0.936183,0.623312,0.602476,0.505203,0.552775,0.773197
2,3,C:\Users\moht2\Documents\Afterimage\finetuning...,C:\Users\moht2\Documents\Afterimage\finetuning...,complete,3.0,0.008312,66c7d56a4f9cb9f5e2375c961ea2e23bd6b3a6d0d1792a...,5.0,0.771117,0.872970,0.931582,0.631052,0.661158,0.489120,0.584465,0.796058


,run,best_epoch,epochs_completed,best_eval_loss,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy
0,1,3.0,5.0,0.008101,0.773842,0.866584,0.942846,0.613749,0.643718
1,2,2.0,4.0,0.007336,0.806540,0.891526,0.936183,0.623312,0.602476
2,3,3.0,5.0,0.008312,0.771117,0.872970,0.931582,0.631052,0.661158


,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy
count,3.000000,3.000000,3.000000,3.000000,3.000000
mean,0.783833,0.877027,0.936870,0.622704,0.635784
std,0.019712,0.012956,0.005663,0.008667,0.030135
min,0.771117,0.866584,0.931582,0.613749,0.602476
max,0.806540,0.891526,0.942846,0.631052,0.661158


,run,adapter_dir,adapter_sha256
0,1,C:\Users\moht2\Documents\Afterimage\finetuning...,caa3918d9a0dee9e76fe75bb0250833d9cb188e44d0959...
1,2,C:\Users\moht2\Documents\Afterimage\finetuning...,5ef24c4e43d6c50546fafb73b1d4034d42ccff0ed06fdf...
2,3,C:\Users\moht2\Documents\Afterimage\finetuning...,66c7d56a4f9cb9f5e2375c961ea2e23bd6b3a6d0d1792a...


Paper reference: accuracy 0.8174; QWK 0.9292; macro ROC-AUC 0.9488.
Compare all completed runs; test metrics were not used to select checkpoints.


## Inspect each adapter's APTOS and external Messidor results

In [7]:
def show_evaluation(report_dir, name="aptos_test"):
    report_dir = Path(report_dir)
    metrics = json.loads((report_dir / f"{name}.summary.json").read_text(encoding="utf-8"))
    columns = ["samples", "accuracy", "qwk", "macro_ovr_auc", "macro_f1", "balanced_accuracy"]
    display(pd.DataFrame([{key: metrics[key] for key in columns}], index=[name]))
    display(pd.DataFrame(metrics["per_class"]))
    matrix = pd.DataFrame(metrics["confusion_matrix"], index=pd.Index(range(5), name="true_grade"), columns=range(5))
    display(matrix)
    try:
        import matplotlib.pyplot as plt
        import numpy as np
    except ImportError:
        print("Numeric reports are complete. Install matplotlib only if you want plots.")
        return metrics
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix.to_numpy(), cmap="Blues")
    fig.colorbar(im, ax=ax)
    ax.set(xticks=range(5), yticks=range(5), xlabel="Predicted grade", ylabel="True grade", title=name)
    for i in range(5):
        for j in range(5):
            value = int(matrix.iloc[i, j])
            ax.text(j, i, str(value), ha="center", va="center", color="white" if value > matrix.to_numpy().max()/2 else "black")
    fig.tight_layout()
    fig.savefig(report_dir / f"{name}.confusion_matrix.png", dpi=180)
    plt.show()
    predictions = pd.read_csv(report_dir / f"{name}.predictions.csv")
    fig, ax = plt.subplots(figsize=(6, 5))
    for grade, label in enumerate(reproduction.LABEL_NAMES):
        truth = predictions["y_true"].to_numpy() == grade
        scores = predictions[f"p{grade}"].to_numpy()
        order = np.argsort(-scores, kind="stable")
        y, s = truth[order], scores[order]
        last = np.r_[np.flatnonzero(np.diff(s)), len(s)-1]
        tp = np.cumsum(y)[last]
        fp = 1 + last - tp
        ax.plot(np.r_[0, fp]/(~truth).sum(), np.r_[0, tp]/truth.sum(),
                label=f"{grade}: {label} (AUC={metrics['per_class'][grade]['auc']:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title=f"{name}: one-vs-rest ROC")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(report_dir / f"{name}.roc.png", dpi=180)
    plt.show()
    return metrics

In [8]:
for entry in batch_results:
    if entry["status"] != "complete":
        continue
    print(f"Run {entry['run']} — APTOS test (367 images)")
    report_dir = Path(entry["run_dir"]) / "evaluation"
    aptos_metrics = show_evaluation(report_dir, "aptos_test")
    assert aptos_metrics["samples"] == 367
    if INCLUDE_EXTERNAL_MESSIDOR:
        print(f"Run {entry['run']} — external Messidor (1,057 images)")
        external_metrics = show_evaluation(report_dir, "messidor_external")
        assert external_metrics["samples"] == 1057

Run 1 — APTOS test (367 images)


,samples,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy
aptos_test,367,0.773842,0.866584,0.942846,0.613749,0.643718


,grade,label,support,precision,recall,f1,auc
0,0,No DR,181,1.000000,0.944751,0.971591,0.997178
1,1,Mild,37,0.388889,0.945946,0.551181,0.949222
2,2,Moderate,100,0.760000,0.570000,0.651429,0.926816
3,3,Severe,19,0.500000,0.157895,0.240000,0.912129
4,4,Proliferative DR,30,0.720000,0.600000,0.654545,0.928882


,0,1,2,3,4
true_grade,,,,,
0,171,10,0,0,0
1,0,35,2,0,0
2,0,39,57,1,3
3,0,1,11,3,4
4,0,5,5,2,18


Numeric reports are complete. Install matplotlib only if you want plots.
Run 1 — external Messidor (1,057 images)


,samples,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy
messidor_external,1057,0.446547,0.561342,0.805607,0.324725,0.32497


... (15 outputs left out)

,samples,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy
messidor_external,1057,0.48912,0.584465,0.796058,0.428807,0.417542


,grade,label,support,precision,recall,f1,auc
0,0,No DR,468,0.648241,0.826923,0.726761,0.798968
1,1,Mild,207,0.194767,0.323671,0.243194,0.488048
2,2,Moderate,290,0.416667,0.103448,0.165746,0.781055
3,3,Severe,71,0.880000,0.309859,0.458333,0.951061
4,4,Proliferative DR,21,0.578947,0.523810,0.550000,0.961160


,0,1,2,3,4
true_grade,,,,,
0,387,78,0,0,3
1,138,67,1,0,1
2,68,187,30,2,3
3,3,9,36,22,1
4,1,3,5,1,11


Numeric reports are complete. Install matplotlib only if you want plots.


## Saved files and restart behavior

```text
artifacts/paper_reproduction/
  data/<fingerprint>/                    original-split CSV exports and dataset manifest
  training_batches/<BATCH_ID>/
    paper_training_configuration.json   fixed values, source/data hashes and package versions
    batch_results.json / .csv            status and metrics for all three runs
    aptos_run_comparison.csv             completed APTOS results
    aptos_repeatability_summary.csv      descriptive mean/std/range
    run_01/                             run_02 and run_03 have the same independent layout
      best_lora/                        best adapter + processor, kept permanently
      checkpoints/                      up to two Trainer checkpoints for this run
      configuration.json
      resolved_training_arguments.json
      run_info.json
      training_complete.json
      training.log
      evaluation.log
      evaluation/                       APTOS and external Messidor predictions/reports
```

Keep the same `BATCH_ID` to recover this batch. Choose a new one to deliberately start another three runs.
Do not delete `best_lora/` or reuse another experiment's run folder. File hashes detect changed adapter files;
different file hashes alone do not quantify numerical differences between tensors.

The new workflow imports shared dataset/sampler definitions from `train_dr_qwen3vl_runsave.py`; it does not
edit that script or the earlier notebook. `paper_reproduction.py` contains the new orchestration and scorer.
The cache/artifact directory is already gitignored. No model weights or patient images belong in the Git commit.

Reference precedence: the paper's APTOS settings and `Qwen3_VL_8B_DR_finetuningAndEvaluation.ipynb`
define this experiment. `Untitled36.ipynb` includes balanced offline augmentation, mixed-data experiments
and patience 8; those are separate experiments and are excluded. `Untitled39.ipynb` contains baseline
and fine-tuned report aggregation, including the paper's 0.8174 / 0.9292 / 0.9488 APTOS results.